In this file we classify each photo of a particular species with either "Invasive", "Native" or "Introduced non-invasive".

To do so, we take the coordinate of each picture and see in which bounding box it falls into (file "Updated_Complete_Region_Dataset.csv"). If it falls into more than one bounding box, we check for which one it is closer to center.

In [24]:
import pandas as pd
import ast
import math

In [26]:
#change to desired path

PLANT_DISTRIBUTION = './support_files/plant_distribution.csv' #this contains the region where a plant is introduced or native
REGION_COORDINATES_DATASET = './support_files/Updated_Complete_Region_Dataset.csv' #this contains the coordinates for the different regions

#manually identified invasive species
INVASIVE_SPECIES = ['lythrum hyssopifolia', 'lythrum salicaria', 'lythrum virgatum']

In [41]:
#import the dataset into a Pandas DataFrame. Avoid doing it every time the function "generate_classification()" is called.

plant_distribution_df = pd.read_csv(PLANT_DISTRIBUTION)
regions_df = pd.read_csv(REGION_COORDINATES_DATASET)

cols = ['introduced', 'native']

#Since when importing the dataframe into pandas lists are evaluated as string, they need to be reverted to their original form
for col in cols:
    plant_distribution_df[col] = plant_distribution_df[col].apply(lambda x : ast.literal_eval(x) if pd.notnull(x) else x)

regions_df['bbox'] = regions_df['bbox'].apply(lambda x : ast.literal_eval(x) if pd.notnull(x) else x)

In [51]:
def generate_classification(lat: float, lon: float, species_name: str, regions_df: pd.DataFrame, plant_distribution_df: pd.DataFrame) -> str:

    """Given the latitude and longitude of a picture and the name of the species (lowercase separated by a ' ',
    e.g. 'lythrum alatum'), return the classification ('native', 'invasive', 'introduced non-invasive')
    for the picture of the species."""

    current_region_index = -1 #indices can never be -1

    for i, row in regions_df.iterrows():

        if row['bbox'][0]<lat<row['bbox'][1] and row['bbox'][2]<lon<row['bbox'][3]: 
                
            distance = math.dist((lat, lon), (row['lat'], row['lon']))

            if current_region_index == -1:
                current_region_index = i
            else:
                if distance < math.dist((lat, lon), (regions_df.at[current_region_index, 'lat'], regions_df.at[current_region_index, 'lon'])):
                    #update the region that best matches the coordinates
                    current_region_index = i

    if current_region_index == -1:
        print("Failed to locate the coordinates into a region")
        return None

    #Now we have the name of the region to be looked at in the "plant_distribution.csv" file
    region_name = regions_df.at[current_region_index, 'Original name']
    print(region_name)

    if region_name in plant_distribution_df.loc[plant_distribution_df['Species']==species_name, 'introduced'].iloc[0]:

        if species_name in INVASIVE_SPECIES:
            classification = 'invasive'
        else:
            classification = 'introduced non-invasive'

    elif region_name in plant_distribution_df.loc[plant_distribution_df['Species']==species_name, 'native'].iloc[0]:
        classification = 'native'

    else:
        print(f"Unable to classify this picture for species {species_name}, the region is not native or introduced")
        return None



    return classification
            

In [56]:
print(generate_classification(39.0026311272,-76.9129070798, 'lythrum salicaria', regions_df, plant_distribution_df))

Maryland
invasive


In [60]:
print(generate_classification(44.0260305,-124.0861278333, 'lythrum portula', regions_df, plant_distribution_df))
print(generate_classification(36.5492033333,-5.3860466667, 'lythrum portula', regions_df, plant_distribution_df))


Oregon
introduced non-invasive
Spain
native


In [61]:
print(generate_classification(37.6238662945,-7.5626955524, 'lythrum borysthenicum', regions_df, plant_distribution_df))

Portugal
native


In [63]:
52.0594631369,29.2682302358
print(generate_classification(52.0594631369,29.2682302358, 'lythrum hyssopifolia', regions_df, plant_distribution_df))
-37.6201569073,143.5902729028
print(generate_classification(-37.6201569073,143.5902729028, 'lythrum hyssopifolia', regions_df, plant_distribution_df))


Belarus
native
Victoria
native
